# Kaggle: huấn luyện 5 mô hình đa-view (MVDet, MVDeTr, SHOT, EarlyBird, TrackTacular)

Sửa **ô cấu hình đầu tiên** rồi chạy lần lượt các ô. Notebook sẽ: lấy mã nguồn, cài venv cho các model đã chọn,
chuẩn bị dataset (WaveTrackNew, Wildtrack hoặc MultiviewX), sinh nhãn drop và pseudo cache khi cần, train + test
từng tổ hợp, rồi gom **log.txt, kết quả test, weight, config** vào thư mục output.

Cần bật **Internet** và **GPU** (Settings → Accelerator / Internet).

In [ ]:
# ============================ THIẾT LẬP THỰC NGHIỆM (chỉ sửa ô này) ============================
DATASET      = "wavetrack"     # "wavetrack" | "wildtrack" | "multiviewx"
# Thư mục gốc dataset, cùng cấu trúc như trên máy: F:\thesis_dataset\WaveTrackNew
# (annotations_positions/, calibrations/, Image_subsets/). Để "" để tự tìm trong /kaggle/input.
DATASET_PATH = "/kaggle/input/wavetracknew/WaveTrackNew"

MODELS       = ["mvdet", "mvdetr", "shot", "earlybird", "tracktacular"]
DROP_LEVELS  = [0, 20, 45, 60]  # 0 = full supervised (baseline); 20/45/60 = % nhãn bị drop
DROP_MODE    = "instance"       # "instance" = drop người trong mỗi frame | "frame" = drop cả frame

# Mỗi phần tử là một cấu hình (loss, pseudo):
#   loss   : "orig" = loss gốc của model (MVDet/SHOT: mse, còn lại: focal) | "aabl" = AABL (brl)
#   pseudo : "off" | "on"  (pseudo label YOLO; tự sinh pseudo cache nếu thiếu; bị bỏ qua khi drop = 0)
CONFIGS = [
    ("orig", "off"),   # loss gốc, không pseudo
    ("aabl", "on"),    # AABL + pseudo label
]

EPOCHS       = None             # None = mặc định của từng model
SEED         = 1                # chỉ áp dụng cho MVDet / MVDeTr / SHOT
SAVE_WEIGHTS = True             # lưu weight (*.pth, *.ckpt) vào output
RERUN_DONE   = False            # True = chạy lại cả tổ hợp đã xong
GPU_ID       = "0"

# --- Nguồn mã ---
REPO_URL        = "https://github.com/duclld1709/multiview-pedestrian-detection.git"
REPO_BRANCH     = "original"
REPO_INPUT_PATH = ""            # nếu mã nằm trong một Kaggle dataset đã attach: đường dẫn tới thư mục chứa run.sh (ưu tiên hơn git)

# --- Output ---
OUTPUT_DIR  = "/kaggle/working/output"
RESUME_FROM = ""                # đường dẫn output của lần chạy trước (attach làm dataset) để bỏ qua các run đã xong
YOLO_WEIGHTS = "yolo26s.pt"     # detector sinh pseudo cache

In [ ]:
import glob, json, os, re, shutil, subprocess, sys, time

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else os.path.abspath("./kaggle_work")
os.makedirs(WORK, exist_ok=True)
REPO = os.path.join(WORK, "original")
OUTPUT_DIR = OUTPUT_DIR if ON_KAGGLE else os.path.join(WORK, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def sh(cmd, cwd=None, check=True, env=None):
    """Chạy lệnh bash, in output ngay khi có."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(["bash", "-lc", cmd], cwd=cwd, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"lệnh lỗi (exit {code}): {cmd}")
    return code

# ---------- 1. mã nguồn ----------
if not os.path.exists(os.path.join(REPO, "run.sh")):
    if REPO_INPUT_PATH:
        hits = glob.glob(os.path.join(REPO_INPUT_PATH, "**", "run.sh"), recursive=True)
        assert hits, f"không thấy run.sh trong {REPO_INPUT_PATH}"
        shutil.copytree(os.path.dirname(sorted(hits, key=len)[0]), REPO)
    else:
        sh(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO}")
assert os.path.exists(os.path.join(REPO, "data_tools", "prepare_wavetrack.py")), (
    "bản mã này chưa có hỗ trợ WaveTrack (data_tools/prepare_wavetrack.py): hãy push/commit nhánh có thay đổi "
    "WaveTrack, hoặc attach mã làm Kaggle dataset và đặt REPO_INPUT_PATH")
print("mã nguồn:", REPO)

# ---------- 2. môi trường (venv riêng cho mỗi model; uv + torch theo GPU) ----------
need_pseudo = any(p == "on" for _, p in CONFIGS) and any(d > 0 for d in DROP_LEVELS)
envs = list(MODELS) + (["tools"] if need_pseudo else [])
env_root = os.environ.get("MVPD_ENV_ROOT") or ("/tmp/mvpd_envs" if ON_KAGGLE else os.path.join(REPO, ".envs"))
missing = [m for m in envs if not (os.path.exists(f"{env_root}/{m}/bin/python") or os.path.exists(f"{env_root}/{m}/Scripts/python.exe"))]
if missing:
    sh(f"bash setup_env.sh {' '.join(missing)}", cwd=REPO)
else:
    print("venv đã có:", envs)

In [ ]:
# ---------- 3. dataset: bản làm việc ghi được (Kaggle /kaggle/input chỉ đọc) ----------
EXPECT = {"wavetrack": "WaveTrackNew", "wildtrack": "Wildtrack", "multiviewx": "MultiviewX"}[DATASET]
src = DATASET_PATH
if not (src and os.path.isdir(src)):
    marker = "calibrations/calibration.json" if DATASET == "wavetrack" else "annotations_positions"
    roots = glob.glob(f"/kaggle/input/**/{marker}", recursive=True) if ON_KAGGLE else []
    assert roots, f"không thấy dataset ở {DATASET_PATH!r}; hãy attach dataset và đặt DATASET_PATH"
    src = os.path.dirname(os.path.dirname(roots[0])) if DATASET == "wavetrack" else os.path.dirname(roots[0])
print("dataset nguồn:", src)

DATA_DIR = os.path.join(REPO, "Data", EXPECT)     # tên thư mục phải chứa "wavetrack" / "wildtrack" / "multiviewx"
if not os.path.isdir(os.path.join(DATA_DIR, "annotations_positions")):
    if DATASET == "wavetrack":
        # điền positionID còn thiếu, bỏ file *.review-backup, symlink ảnh
        sh(f"python3 data_tools/prepare_wavetrack.py --src '{src}' --dst '{DATA_DIR}' --images link", cwd=REPO)
    else:
        os.makedirs(DATA_DIR, exist_ok=True)
        for name in os.listdir(src):
            target = os.path.join(DATA_DIR, name)
            if name == "Image_subsets":
                os.symlink(os.path.join(src, name), target, target_is_directory=True)  # ảnh: symlink cho nhẹ
            elif os.path.isdir(os.path.join(src, name)):
                shutil.copytree(os.path.join(src, name), target)
            else:
                shutil.copy2(os.path.join(src, name), target)
else:
    print("dataset làm việc đã có:", DATA_DIR)

# nhãn drop (train split) cho các mức > 0; cần numpy của Python hệ thống
sim_mode = "frame" if DROP_MODE == "frame" else "instance"
prefix = "frame_drop" if sim_mode == "frame" else "drop"
todo = [d for d in DROP_LEVELS if d > 0 and not os.path.isdir(os.path.join(DATA_DIR, "drop_annotations", f"{prefix}_{d}"))]
if todo:
    settings = " ".join(f"drop{d}" for d in todo)
    sh(f"python3 data_tools/simulate_dropped_annotations.py -d {EXPECT} -r Data -s {settings} --mode {sim_mode} --seed 1", cwd=REPO)
print("sẵn sàng:", DATA_DIR)

In [ ]:
# ---------- 4. train + test, gom artifact vào OUTPUT_DIR ----------
# experiment_logs (marker "đã xong", log từng run, summary) nằm trong output để resume được.
exp_logs = os.path.join(OUTPUT_DIR, "experiment_logs")
os.makedirs(exp_logs, exist_ok=True)
link = os.path.join(REPO, "experiment_logs")
if os.path.islink(link):
    os.unlink(link)
elif os.path.isdir(link):
    shutil.copytree(link, exp_logs, dirs_exist_ok=True); shutil.rmtree(link)
os.symlink(exp_logs, link, target_is_directory=True)

if RESUME_FROM and os.path.isdir(RESUME_FROM):
    shutil.copytree(RESUME_FROM, OUTPUT_DIR, dirs_exist_ok=True)
    print("đã nạp kết quả cũ từ", RESUME_FROM)

with open(os.path.join(OUTPUT_DIR, "run_config.json"), "w") as f:
    json.dump(dict(DATASET=DATASET, DATASET_PATH=DATASET_PATH, MODELS=MODELS, DROP_LEVELS=DROP_LEVELS,
                   DROP_MODE=DROP_MODE, CONFIGS=CONFIGS, EPOCHS=EPOCHS, SEED=SEED, SAVE_WEIGHTS=SAVE_WEIGHTS,
                   REPO_BRANCH=REPO_BRANCH, started=time.strftime("%Y-%m-%d %H:%M:%S")), f, indent=2)

failed_total = 0
for loss, pseudo in CONFIGS:
    cmd = ["bash", "run_experiments.sh",
           "--dataset", DATASET, "--data-path", DATA_DIR,
           "--models", ",".join(MODELS), "--drops", ",".join(map(str, DROP_LEVELS)),
           "--drop-mode", DROP_MODE, "--losses", loss, "--pseudo", pseudo,
           "--collect", OUTPUT_DIR, "--no-gen-drops", "--weights", YOLO_WEIGHTS, "--gpu", str(GPU_ID)]
    if EPOCHS: cmd += ["--epochs", str(EPOCHS)]
    if SEED is not None: cmd += ["--seed", str(SEED)]
    if not SAVE_WEIGHTS: cmd += ["--no-weights"]
    if RERUN_DONE: cmd += ["--rerun"]
    print(f"\n########## cấu hình: loss={loss}  pseudo={pseudo} ##########", flush=True)
    failed_total += sh(" ".join(map(__import__("shlex").quote, cmd)), cwd=REPO, check=False) != 0
print("\nxong tất cả cấu hình;", "có run lỗi, xem experiment_logs/summary_*.tsv" if failed_total else "không có lỗi")

In [ ]:
# ---------- 5. tổng hợp kết quả ----------
import pandas as pd

NUM = r"(-?\d+(?:\.\d+)?)"
def parse_metrics(run_dir):
    """MODA/MODP/precision/recall cuối cùng từ log.txt (MVDet, MVDeTr, SHOT) hoặc console.log (EarlyBird, TrackTacular)."""
    pat = re.compile(rf"moda: {NUM}%, modp: {NUM}%, precision: {NUM}%, recall: {NUM}%")
    for log in glob.glob(os.path.join(run_dir, "**", "log.txt"), recursive=True):
        hits = pat.findall(open(log, errors="ignore").read())
        if hits:
            return dict(zip(["moda", "modp", "precision", "recall"], map(float, hits[-1])))
    console = os.path.join(run_dir, "console.log")
    if os.path.exists(console):
        text = open(console, errors="ignore").read()
        out = {}
        for key in ["moda", "modp", "precision", "recall"]:
            hits = re.findall(rf"detect/{key}\W+{NUM}", text)
            if hits:
                out[key] = float(hits[-1])
        if out:
            return out
    return {}

rows = []
for run_dir in sorted(glob.glob(os.path.join(OUTPUT_DIR, "*_*_*_*_*"))):
    if not os.path.isdir(run_dir) or os.path.basename(run_dir) == "experiment_logs":
        continue
    tag = os.path.basename(run_dir)
    m = re.fullmatch(r"(?P<dataset>[a-z]+)_(?P<model>[a-z]+)_(?P<drop>full|instance\d+|frame\d+)_(?P<loss>orig|aabl)_(?P<pl>pl|nopl)", tag)
    if not m:
        continue
    info = m.groupdict()
    info["pseudo"] = info.pop("pl") == "pl"
    info.update(parse_metrics(run_dir)); info["tag"] = tag
    rows.append(info)
df = pd.DataFrame(rows)
if len(df):
    order = [c for c in ["dataset", "model", "drop", "loss", "pseudo", "moda", "modp", "precision", "recall", "tag"] if c in df]
    df = df[order].sort_values(["model", "drop", "loss", "pseudo"]).reset_index(drop=True)
    df.to_csv(os.path.join(OUTPUT_DIR, "summary.csv"), index=False)
    display(df)
else:
    print("chưa có kết quả nào trong", OUTPUT_DIR)
print("\ncấu trúc output:")
sh(f"cd '{OUTPUT_DIR}' && du -sh * 2>/dev/null | sort -h | tail -20", check=False)

## Cấu trúc thư mục output

```
output/
├── run_config.json              # cấu hình của lần chạy
├── summary.csv                  # bảng MODA / MODP / precision / recall cho mọi run
├── experiment_logs/             # log console từng run, summary_*.tsv, marker "đã xong"
└── <dataset>_<model>_<drop>_<loss>_<pl|nopl>/
    ├── command.txt              # đúng lệnh đã chạy
    ├── console.log              # toàn bộ stdout của train + test
    ├── <thư mục log>/           # MVDet/MVDeTr/SHOT: log.txt, test.txt, MultiviewDetector.pth, ...
    └── lightning_logs_version_N # EarlyBird/TrackTacular: config.yaml, checkpoints/*.ckpt, kết quả test
```

Ghi chú:
- Session Kaggle có giới hạn thời gian; nếu bị ngắt, lưu output rồi chạy lại với `RESUME_FROM` trỏ tới output cũ
  để bỏ qua các run đã xong.
- Dung lượng output tối đa của Kaggle là 20 GB; đặt `SAVE_WEIGHTS = False` nếu không cần weight.